In [ ]:
import sys
sys.path.insert(0, '../Week-5-6-7-8')

import json
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR, NuSVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from helpers.modeling import identify_column_types, create_preprocessor, evaluate_model
from uncertainty import conformal_quantile, build_intervals, calibration_table

with open('../Week-5-6-7-8/results.json', 'r') as f:
    results = json.load(f)

df = pd.read_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv")

In [ ]:
target_col = 'cs_28d'
pub_col    = 'paper_reference'
threshold  = 5
alpha      = 0.90

X = df.drop(columns=[target_col, pub_col])
y = df[target_col]

# Preprocessor template — cloned fresh for every pipeline fit inside the loop
numerical_cols, one_hot_cols, k_fold_cols = identify_column_types(X)
preprocessor_template = create_preprocessor(numerical_cols, one_hot_cols, k_fold_cols,
                                             handle_unknown='ignore')

model_configs = [
    ('knn',   KNeighborsRegressor, 'knn'),
    ('svr',   SVR,                 'svr'),
    ('nusvr', NuSVR,               'nusvr'),
]

selected_pubs = [
    pub for pub, grp in df.groupby(pub_col) if len(grp) >= threshold
]
print(f"Publications with >= {threshold} rows: {len(selected_pubs)} / {df[pub_col].nunique()}")

In [ ]:
all_intervals = []

for pub_id in tqdm(selected_pubs):
    test_mask = df[pub_col] == pub_id
    X_te = X[test_mask]
    y_te = y[test_mask]

    X_tr_all = X[~test_mask]
    y_tr_all = y[~test_mask]

    # Hold out 15 % of training as calibration set for conformal interval
    X_tr_fit, X_cal, y_tr_fit, y_cal = train_test_split(
        X_tr_all, y_tr_all, test_size=0.15, random_state=42
    )

    for model_name, model_cls, param_key in model_configs:
        params = results["best_params"]["recoded_50"][param_key]
        pipeline = Pipeline([
            ('preprocessor', clone(preprocessor_template)),
            ('model',        model_cls()),
        ])
        pipeline.set_params(**params)
        pipeline.fit(X_tr_fit, y_tr_fit)

        q = conformal_quantile(pipeline, X_cal, y_cal, alpha)
        row_df = build_intervals(
            pipeline, X_te, y_te,
            pd.Series([pub_id] * len(X_te), name='publication'),
            q, X_tr_fit
        )
        row_df['model'] = model_name
        all_intervals.append(row_df)

logo_intervals = pd.concat(all_intervals, ignore_index=True)
print(f"Total predicted rows: {len(logo_intervals)}")
logo_intervals.head()

In [ ]:
# Pooled metrics per model across all LOGO folds
for model_name in ['knn', 'svr', 'nusvr']:
    df_m = logo_intervals[logo_intervals['model'] == model_name]
    rmse = np.sqrt((df_m['residual'] ** 2).mean())
    mae  = df_m['residual'].mean()
    cov  = ((df_m['true'] >= df_m['lower']) & (df_m['true'] <= df_m['upper'])).mean()
    w    = (df_m['upper'] - df_m['lower']).mean()
    print(f"{model_name.upper():6s} | RMSE={rmse:.2f}  MAE={mae:.2f}  Coverage={cov:.2%}  Avg Width={w:.2f}")

In [ ]:
# Per-publication calibration table — KNN
cal_logo_knn = calibration_table(
    logo_intervals[logo_intervals['model'] == 'knn']
)
cal_logo_knn

In [ ]:
# Per-publication calibration table — SVR
cal_logo_svr = calibration_table(
    logo_intervals[logo_intervals['model'] == 'svr']
)
cal_logo_svr

In [ ]:
# Per-publication calibration table — NuSVR
cal_logo_nusvr = calibration_table(
    logo_intervals[logo_intervals['model'] == 'nusvr']
)
cal_logo_nusvr

In [ ]:
poor_coverage_threshold = 0.90

poor_coverage_logo = pd.concat([
    cal_logo_knn.assign(model='knn'),
    cal_logo_svr.assign(model='svr'),
    cal_logo_nusvr.assign(model='nusvr'),
])
poor_coverage_logo = (
    poor_coverage_logo[poor_coverage_logo['coverage'] < poor_coverage_threshold]
    .sort_values('coverage')
    [['model', 'publication', 'n_rows', 'coverage', 'interval_width', 'mean_residual', 'mean_distance']]
    .reset_index(drop=True)
)

print(f"Publications below {poor_coverage_threshold:.0%} coverage: {poor_coverage_logo['publication'].nunique()} unique, {len(poor_coverage_logo)} model-publication pairs")
poor_coverage_logo

In [ ]:
import matplotlib.pyplot as plt

all_cal_logo = pd.concat([
    cal_logo_knn.assign(model='knn'),
    cal_logo_svr.assign(model='svr'),
    cal_logo_nusvr.assign(model='nusvr'),
], ignore_index=True)

colors = {'knn': 'steelblue', 'svr': 'tomato', 'nusvr': 'seagreen'}

fig, ax = plt.subplots(figsize=(9, 6))

for model_name, grp in all_cal_logo.groupby('model'):
    ax.scatter(grp['mean_distance'], grp['mean_residual'],
               label=model_name.upper(), color=colors[model_name],
               alpha=0.7, edgecolors='white', linewidths=0.4, s=60)

for model_name, grp in all_cal_logo.groupby('model'):
    for _, row in grp.nlargest(5, 'mean_residual').iterrows():
        ax.annotate(row['publication'].replace('-Research', '').replace('-data', ''),
                    xy=(row['mean_distance'], row['mean_residual']),
                    fontsize=7, color=colors[model_name],
                    xytext=(4, 2), textcoords='offset points')

ax.set_xlabel('Mean Distance to Training (5-NN)', fontsize=11)
ax.set_ylabel('Mean Absolute Residual (MPa)', fontsize=11)
ax.set_title('Distance vs Residual per Publication — LOGO', fontsize=12)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()